# 01 · Quickstart

Language-Mass Precision compares a process model with an event log through the language each one defines:

$$\mathrm{LMP}_\mu(L, M) = \frac{\mu(\tilde L \cap \mathcal L(M))}{\mu(\mathcal L(M))}$$

$\tilde L$ is the set of distinct variants in the log, $\mathcal L(M)$ the model language, and $\mu$ a fixed full-support probability mass over all traces $\Sigma^*$. LMP is the share of the model language's mass that the log actually supports. A model that admits nothing unseen scores 1; a model whose language is dominated by unseen behaviour scores close to 0.

This notebook covers the three ways to call the library and how to read its result object.

In [1]:
import os, warnings
os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "False"
warnings.filterwarnings("ignore", category=DeprecationWarning)

import language_mass_precision as lmp
import pm4py
import pandas as pd
pd.set_option("display.precision", 6)

## A first score

Logs can be a PM4Py DataFrame, a PM4Py `EventLog`, a `{trace: count}` mapping, or a plain list of traces. Models can be `(net, im, fm)`, a process tree or a BPMN diagram.

In [2]:
log = [("a", "b", "c"), ("a", "d", "c"), ("a", "b", "c")]
model = pm4py.parse_process_tree("->('a', X('b', 'd', 'e'), 'c')")

result = lmp.precision_language_mass(log, model)
result

LMPResult(status=exact, value=0.666667, certification='full_language_finite_sum')

The model admits `a e c`, which the log never shows. Under the default prior (geometric length prior calibrated on the log, uniform activity prior) all three traces have length 3 and therefore the same mass, so LMP is exactly 2/3.

The result is an `LMPResult`. Its `status` says what kind of answer was produced:

| status | meaning |
|---|---|
| `exact` | full-language value, computed exactly |
| `interval` | certified interval `[lower, upper]` containing the full-language value |
| `bounded` | value over traces of length ≤ K only (playout baseline), no certificate |
| `incomplete` | the language could not be compiled within the state/time limits |
| `undefined` | the model language is empty |

In [3]:
print("status       :", result.status)
print("value        :", result.value)
print("certification:", result.certification)
print("float(result):", float(result))
print("imprecision  :", result.imprecision)
pd.Series(result.to_dict())

status       : exact
value        : 0.6666666666666672
certification: full_language_finite_sum
float(result): 0.6666666666666672
imprecision  : 0.3333333333333328


status                                       exact
value                                     0.666667
lower                                     0.666667
upper                                     0.666667
certification             full_language_finite_sum
numerator_mass                            0.001514
log_numerator_mass                       -6.493227
denominator_lower                          0.00227
denominator_upper                          0.00227
accepted_variants                                2
rejected_variants                                0
truncation_k                                     3
dfa_states                                       4
reachability_states                              4
reason                                        None
elapsed_seconds                           0.000833
prior_length_prior                       geometric
prior_length_parameter                    0.840896
prior_activity_prior                       uniform
prior_alphabet_size            

## Same answer, any input format

In [4]:
frame = lmp.traces_to_dataframe(log)          # PM4Py-format DataFrame
event_log = pm4py.convert_to_event_log(frame)
net, im, fm = pm4py.convert_to_petri_net(model)
bpmn = pm4py.convert_to_bpmn(model)

pd.DataFrame({
    "list + tree":        [lmp.precision_language_mass(log, model).value],
    "DataFrame + net":    [lmp.precision_language_mass(frame, net, im, fm).value],
    "EventLog + BPMN":    [lmp.precision_language_mass(event_log, bpmn).value],
    "counts + tree":      [lmp.precision_language_mass({("a","b","c"): 2, ("a","d","c"): 1}, model).value],
})

,list + tree,DataFrame + net,EventLog + BPMN,counts + tree
0,0.666667,0.666667,0.666667,0.666667


Frequencies do not matter: LMP looks only at the log *support*. Adding a thousand copies of `a b c` changes nothing.

## Three interfaces

**1 · Simplified functions** in the style of `pm4py.precision_alignments`:

In [5]:
lmp.precision_language_mass(frame, net, im, fm, length_prior="power_law")

LMPResult(status=exact, value=0.666667, certification='full_language_finite_sum')

**2 · A scikit-learn style estimator.** `fit` reads the log, `compile` turns a model into a DFA once, `score` evaluates it. Hyperparameters live in the constructor and can be changed with `set_params` without refitting or recompiling.

In [6]:
estimator = lmp.LanguageMassPrecision(length_prior="geometric")
estimator.fit(frame)
print(estimator)
print("variants   :", sorted(estimator.variants_))
print("alphabet   :", sorted(estimator.alphabet_))
print("median len :", estimator.median_length_)
print("length prior:", estimator.length_prior_)

compiled = estimator.compile(model, name="choice")
print(compiled)
estimator.score(compiled)

LanguageMassPrecision()
variants   : [('a', 'b', 'c'), ('a', 'd', 'c')]
alphabet   : ['a', 'b', 'c', 'd']
median len : 3.0
length prior: GeometricLengthPrior(lam=0.8408964152537145, name='geometric')
CompiledModel('choice', status='ok', dfa_states=4, reachability_states=4, labels=5)


LMPResult(status=exact, value=0.666667, certification='full_language_finite_sum')

**3 · The PM4Py algorithm interface**, with `Variants` and `Parameters` enums:

In [7]:
from language_mass_precision.algo.precision import algorithm as lmp_algorithm

lmp_algorithm.apply(
    frame, net, im, fm,
    variant=lmp_algorithm.Variants.EXACT,
    parameters={lmp_algorithm.Parameters.LENGTH_PRIOR: "power_law",
                lmp_algorithm.Parameters.LENGTH_PARAMETER: 2.0},
)

LMPResult(status=exact, value=0.666667, certification='full_language_finite_sum')

## Ranking several models

`rank_models` scores every model under **one shared prior**. The alphabet is the union of the log activities and all model-language labels, so the values are directly comparable.

In [8]:
models = {
    "exact":   pm4py.parse_process_tree("->('a', X('b', 'd'), 'c')"),
    "extra_e": model,
    "loop":    pm4py.parse_process_tree("->('a', *(X('b', 'd'), tau), 'c')"),
    "flower":  pm4py.parse_process_tree("*(tau, X('a', 'b', 'c', 'd', 'e'))"),
}
lmp.rank_models(log, models)[["model", "rank", "status", "value", "lower", "upper", "certification", "dfa_states"]]

,model,rank,status,value,lower,upper,certification,dfa_states
0,exact,1,exact,1.000000,1.000000,1.000000,full_language_finite_sum,4
1,extra_e,2,exact,0.666667,0.666667,0.666667,full_language_finite_sum,4
2,loop,3,exact,0.663641,0.663641,0.663641,full_language_sparse_solve,4
3,flower,4,exact,0.001514,0.001514,0.001514,full_language_sparse_solve,2


Next: [02 · Controlled examples](02_controlled_examples.ipynb) walks through the witness cases from the paper.